# Exploración y Análisis de ERA5 · Agregación de series temporales de temperatura (`t2m`) · Chile 2000-2025

Proyecto de análisis que parte de la **temperatura del aire a 2 m diaria** (`t2m_media`, `t2m_min`, `t2m_max`, °C) generada en
`era5_extraccion_diario.ipynb` y la agrega en el **tiempo** (mensual, estacional, anual),
calcula **índices de extremos**, **climatologías y anomalías**, y la agrega en el **espacio** (zonas de Chile y ciudades).

| Sección | Contenido |
|---|---|
| 1 | Configuración: imports, parámetros y cluster Dask |
| 2 | Lectura, validación y máscara de Chile |
| 3 | Agregación temporal: mensual, estacional, anual |
| 4 | Índices de temperatura (tipo ETCCDI) y grados-día |
| 5 | Climatologías y anomalías |
| 6 | Agregación espacial: zonas y ciudades |
| 7 | Cómputo único y guardado en NetCDF |
| 8 | Validación (coherencia de promedios) |
| 9 | Visualizaciones |
| 10 | Tendencias |
| 11 | Tabla resumen y exportación a CSV |

**Reglas de agregación**
- La temperatura se **promedia** (°C/día → media mensual, estacional, anual); nunca se suma. Los extremos (TXx, TNn) usan máximo/mínimo y los conteos de días (heladas, días de verano) se suman.
- Se trabaja con tres variables diarias: `tmedia` (media de 24 h), `tmax`, `tmin`, más la **amplitud térmica diaria** `dtr = tmax - tmin`.
- Solo se conservan períodos **completos** (todos sus días presentes). Un mes con días faltantes sesgaría la media hacia los días presentes.
- El día ERA5 va de 00 a 24 h **UTC** (≈ 20-20 h hora local de Chile); la `tmin` puede caer en la madrugada del día calendario siguiente. Para climatología y tendencias no importa, pero conviene tenerlo presente al comparar con estaciones.
- No se calcula año hidrológico: para temperatura no tiene sentido físico (el año calendario ya deja el invierno austral completo dentro de un año).
- Las secciones 2-6 solo construyen grafos perezosos de Dask; el cómputo ocurre en la sección 7 (un cómputo por producto).

## 1. Configuración inicial

### Imports
Se importan todos los paquetes al inicio, ordenados alfabéticamente por módulo.

Requisitos: `xarray dask distributed netCDF4 scipy matplotlib geopandas shapely` y, muy recomendado, **`flox`** (hace que los `groupby` sobre arreglos Dask sean mucho más rápidos):
`pip install flox`.

In [ ]:
import dask
from dask.distributed import Client, get_client
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy.special import stdtr
import shapely
import xarray as xr

### Parámetros

Todo lo ajustable está aquí. La ruta base puede sobrescribirse con la variable de entorno `ERA5_DIR`.
Los umbrales de los índices siguen las definiciones ETCCDI (heladas < 0 °C, días de verano > 25 °C, noches tropicales > 20 °C).

In [ ]:
# --- Rutas -------------------------------------------------------------------
DIR_BASE = Path(os.environ.get('ERA5_DIR', r'D:\Documents\Proyectos\era5'))
ARCHIVO_T2M = DIR_BASE / 'diario_era5_chile' / 't2m_diario_2000_2025.nc'
DIR_SALIDA = DIR_BASE / 'agregados_t2m'
DIR_FIGURAS = DIR_SALIDA / 'figuras'
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

# --- Datos y período ---------------------------------------------------------
# nombre en el archivo -> nombre corto usado en el notebook (°C, salida de era5_extraccion_diario.ipynb)
VARIABLES = {'t2m_media': 'tmedia', 't2m_max': 'tmax', 't2m_min': 'tmin'}
VARS_T = ['tmedia', 'tmax', 'tmin', 'dtr']   # dtr = tmax - tmin (amplitud térmica diaria)
ANIO_INI, ANIO_FIN = 2000, 2025

# --- Umbrales de los índices --------------------------------------------------
UMBRAL_HELADA = 0.0             # °C: helada (FD, TN < 0) y día de hielo (ID, TX < 0)
UMBRAL_SU = 25.0                # °C: día de verano (SU, TX > 25)
UMBRAL_TR = 20.0                # °C: noche tropical (TR, TN > 20)
PERCENTIL_CALIDO = 0.90         # percentil de TX para días cálidos (TX90p) y racha cálida
PERCENTIL_FRIO = 0.10           # percentil de TN para noches frías (TN10p)
BASE_GDD = 10.0                 # °C: base de grados-día de crecimiento (vid, frutales, maíz)
BASE_HDD = 18.0                 # °C: base de grados-día de calefacción

# --- Calendario ----------------------------------------------------------------
ESTACION_POR_MES_INICIO = {12: 'DJF', 3: 'MAM', 6: 'JJA', 9: 'SON'}
ORDEN_ESTACIONES = ['DJF', 'MAM', 'JJA', 'SON']
NOMBRES_MES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

# --- Espacio ---------------------------------------------------------------------
USAR_MASCARA_CHILE = True       # True: solo celdas cuyo centro cae dentro de Chile
ARCHIVO_REGIONES = Path(os.environ.get(
    'ERA5_REGIONES',
    r'D:\Documents\adm_boundaries_overture\chile_limites_administrativos\chile_region.geojson'))
DIBUJAR_LIMITES_REGIONALES = True   # True: dibuja los límites de las 16 regiones; False: solo el contorno del país

# Zonas por banda de latitud (norte, sur), en °S negativos
ZONAS = {
    'Norte Grande': (-17.0, -26.0),
    'Norte Chico':  (-26.0, -32.0),
    'Centro':       (-32.0, -37.0),
    'Sur':          (-37.0, -43.0),
    'Austral':      (-43.0, -56.5),
}
NOMBRE_TOTAL = 'Chile'
ZONA_HEATMAP = 'Centro'

# Ciudades (lat, lon): se toma la celda ERA5 más cercana
CIUDADES = {
    'Arica':        (-18.48, -70.32),
    'Antofagasta':  (-23.65, -70.40),
    'La Serena':    (-29.90, -71.25),
    'Santiago':     (-33.45, -70.67),
    'Concepción':   (-36.83, -73.05),
    'Temuco':       (-38.74, -72.60),
    'Valdivia':     (-39.81, -73.25),
    'Puerto Montt': (-41.47, -72.94),
    'Punta Arenas': (-53.16, -70.91),
}

### Cluster de Dask

Cluster local en proceso con pocos hilos (los datos ya están en disco, no hay red de por medio).
Se cierra cualquier cluster previo por si la celda se vuelve a ejecutar. Docs: [`dask.distributed.Client`](https://distributed.dask.org/en/stable/client.html).

In [ ]:
try:
    get_client().close()
    print('Cluster previo detectado y cerrado.')
except ValueError:
    pass

client = Client(processes=False, n_workers=1, threads_per_worker=8)
client

### Estilo de gráficos

Ejes sobrios, grilla tenue, un solo eje Y por gráfico.
Temperaturas absolutas → escala `RdYlBu_r` (convención frío-azul / cálido-rojo); anomalías y tendencias → par divergente azul/rojo (`RdBu_r`) con centro neutro;
conteos de días → escala secuencial de un solo tono.

In [ ]:
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
})
COLOR_TMAX, COLOR_TMEDIA, COLOR_TMIN = '#B2182B', '#444444', '#2166AC'
COLOR_POS, COLOR_NEG = '#B2182B', '#2166AC'   # anomalía cálida / fría (extremos de RdBu)
COLOR_TEXTO = '#444444'
CMAP_T = 'RdYlBu_r'


def guardar_figura(fig, nombre):
    fig.savefig(DIR_FIGURAS / f'{nombre}.png')

## 2. Lectura, validación y máscara de Chile

### Apertura perezosa
El archivo se abre con **un solo bloque en el tiempo** y bloques de latitud: así los percentiles, las rachas y los `resample`
trabajan sobre la serie completa de cada celda sin tener que reagrupar. Las variables se renombran a `tmedia`, `tmax`, `tmin`.
Docs: [`xarray.open_dataset`](https://docs.xarray.dev/en/stable/generated/xarray.open_dataset.html).

In [ ]:
ds = xr.open_dataset(ARCHIVO_T2M, chunks={'time': -1, 'latitude': 32, 'longitude': -1})
t2m_raw = ds[list(VARIABLES)].rename(VARIABLES)

print(ARCHIVO_T2M)
for v in t2m_raw.data_vars:
    print(f'Variable: {v:7s} | unidades: {t2m_raw[v].attrs.get("units")} | dims: {dict(t2m_raw[v].sizes)}')
print(f'Tiempo: {str(t2m_raw.time.values[0])[:10]} -> {str(t2m_raw.time.values[-1])[:10]}')
print(f'Latitud: {float(t2m_raw.latitude.min()):.2f}..{float(t2m_raw.latitude.max()):.2f} | '
      f'Longitud: {float(t2m_raw.longitude.min()):.2f}..{float(t2m_raw.longitude.max()):.2f}')
print(f'Tamaño en memoria si se cargara: {t2m_raw.nbytes / 1e6:.0f} MB | chunks: {t2m_raw.tmedia.data.chunksize}')

### Chequeos de calidad

Se verifica: días faltantes o fuera del período, rango físico plausible (−60..+50 °C), celdas con NaN y la coherencia diaria `tmin ≤ tmedia ≤ tmax`.
Los días faltantes no se rellenan: los períodos que los contienen se descartan en la sección 3.
Docs: [`pandas.date_range`](https://pandas.pydata.org/docs/reference/api/pandas.date_range.html), [`dask.compute`](https://docs.dask.org/en/stable/api.html#dask.compute).

In [ ]:
tiempos = pd.DatetimeIndex(t2m_raw.time.values)
esperados = pd.date_range(f'{ANIO_INI}-01-01', f'{ANIO_FIN}-12-31', freq='D')
faltan = esperados.difference(tiempos)
sobran = tiempos.difference(esperados)
assert tiempos.is_monotonic_increasing and not tiempos.has_duplicates, 'Eje de tiempo desordenado o con duplicados'

estadisticos = {v: (t2m_raw[v].min(), t2m_raw[v].max(), t2m_raw[v].isnull().sum()) for v in t2m_raw.data_vars}
incoherentes = ((t2m_raw.tmin > t2m_raw.tmedia + 1e-4) | (t2m_raw.tmedia > t2m_raw.tmax + 1e-4)).sum()
estadisticos, incoherentes = dask.compute(estadisticos, incoherentes)

print(f'Días esperados: {len(esperados)} | presentes: {len(tiempos)} | faltan: {len(faltan)} | fuera de período: {len(sobran)}')
if len(faltan):
    print('  Primeros faltantes:', [d.strftime('%Y-%m-%d') for d in faltan[:5]])
for v, (minimo, maximo, n_nan) in estadisticos.items():
    print(f'{v:7s} mínimo: {float(minimo):7.2f} °C | máximo: {float(maximo):6.2f} °C | valores NaN: {int(n_nan)}')
    if float(minimo) < -60 or float(maximo) > 50:
        print(f'  AVISO: {v} fuera del rango físico plausible.')
print(f'Celdas-día con tmin > tmedia o tmedia > tmax: {int(incoherentes)}')

### Límites de Chile: máscara y contorno para los mapas

ERA5 entrega la caja completa (océano + Argentina). Se usa el GeoJSON de regiones (`chile_region.geojson`, Overture Maps: 16 regiones terrestres, EPSG:4326) para:

1. **Enmascarar** las celdas cuyo centro cae fuera del país: `shapely.intersects_xy` prueba todos los centros de celda contra el polígono, en una sola llamada vectorizada.
2. **Dibujar el contorno** (y, opcionalmente, los límites regionales) sobre cada mapa.

Antes se recorta la geometría a la caja de ERA5 con [`GeoSeries.clip_by_rect`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.clip_by_rect.html)
(deja fuera Isla de Pascua y Juan Fernández, que no están en la grilla) y para dibujar se simplifica con [`simplify`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.simplify.html)
(el archivo tiene ~20 MB de costa muy detallada; a 0,25° no se ve la diferencia).
Lectura: [`geopandas.read_file`](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html), máscara: [`shapely.intersects_xy`](https://shapely.readthedocs.io/en/stable/reference/shapely.intersects_xy.html).

Aquí también se añade la amplitud térmica diaria `dtr = tmax - tmin`.
`t2m_raw` (sin máscara) se conserva para las **ciudades costeras**, cuya celda más cercana puede caer en el mar.

In [ ]:
mascara_chile = None
contorno = None   # líneas para dibujar sobre los mapas (GeoSeries); None = no dibujar

if USAR_MASCARA_CHILE:
    try:
        regiones = gpd.read_file(ARCHIVO_REGIONES).to_crs('EPSG:4326')
        paso = 0.25   # se amplía la caja medio píxel para no perder celdas de borde
        x0, x1 = float(t2m_raw.longitude.min()) - paso, float(t2m_raw.longitude.max()) + paso
        y0, y1 = float(t2m_raw.latitude.min()) - paso, float(t2m_raw.latitude.max()) + paso

        # Recorte a la caja de ERA5 y descarte de regiones que quedan fuera (p. ej. Isla de Pascua)
        geoms = regiones.geometry.clip_by_rect(x0, y0, x1, y1)
        regiones_caja = regiones.assign(geometry=geoms)[~geoms.is_empty].reset_index(drop=True)
        pais = regiones_caja.geometry.union_all()
        shapely.prepare(pais)

        # ¿el centro de cada celda cae dentro (o en el borde) del país?
        lon2d, lat2d = np.meshgrid(t2m_raw.longitude.values, t2m_raw.latitude.values)
        dentro = shapely.intersects_xy(pais, lon2d, lat2d)
        mascara_chile = xr.DataArray(dentro, dims=('latitude', 'longitude'),
                                     coords={'latitude': t2m_raw.latitude, 'longitude': t2m_raw.longitude})

        # Geometría liviana para los mapas
        if DIBUJAR_LIMITES_REGIONALES:
            contorno = regiones_caja.geometry.simplify(0.005).boundary
        else:
            contorno = gpd.GeoSeries([shapely.simplify(pais, 0.005).boundary], crs='EPSG:4326')
        print(f'Regiones leídas: {len(regiones)} | dentro de la caja ERA5: {len(regiones_caja)}')
    except Exception as exc:   # archivo ausente, CRS inválido, etc.
        print(f'AVISO: no se pudo construir la máscara desde {ARCHIVO_REGIONES} ({exc!r}). Se usa toda la caja ERA5.')

if mascara_chile is None:
    mascara_chile = xr.ones_like(t2m_raw.tmedia.isel(time=0, drop=True), dtype=bool)

print(f'Celdas dentro de Chile: {int(mascara_chile.sum())} de {mascara_chile.size}')

t2m_raw = t2m_raw.assign(dtr=t2m_raw.tmax - t2m_raw.tmin)
t2m = t2m_raw.where(mascara_chile)   # perezoso

## 3. Agregación temporal

### Funciones auxiliares

- `bins_completos(tiempos, freq)`: devuelve las etiquetas de los períodos cuyo número de días observados es igual al esperado por calendario.
  Así se descartan el primer/último período parcial (p. ej. el `DJF` que empieza en dic-1999) y cualquier período con días faltantes.
- `remuestrear(obj, freq, como)`: media (por defecto), máximo, mínimo o suma por período con [`Dataset.resample`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.resample.html),
  conservando solo períodos completos. La suma (para conteos de días) usa `min_count=1` para que las celdas enmascaradas sigan siendo NaN en vez de 0.

In [ ]:
def bins_completos(tiempos, freq):
    """Etiquetas (inicio de período) de los períodos de `freq` con todos sus días presentes."""
    obs = pd.Series(1, index=pd.DatetimeIndex(tiempos)).resample(freq).sum()
    ultimo_dia = obs.index[-1] + pd.tseries.frequencies.to_offset(freq) - pd.Timedelta(days=1)
    esperado = pd.Series(1, index=pd.date_range(obs.index[0], ultimo_dia)).resample(freq).sum()
    return obs.index[obs.values == esperado.reindex(obs.index).values]


def remuestrear(obj, freq, como='mean'):
    """Agrega `obj` (DataArray o Dataset) por período (`mean`, `max`, `min` o `sum`) y deja solo los períodos completos."""
    r = obj.resample(time=freq)
    out = r.sum(min_count=1) if como == 'sum' else getattr(r, como)()
    return out.sel(time=bins_completos(obj.time.values, freq))


def etiquetar_estacion(obj):
    """Añade la coordenada `estacion` (DJF/MAM/JJA/SON) a una serie trimestral que empieza en dic/mar/jun/sep."""
    etiquetas = [ESTACION_POR_MES_INICIO[m] for m in obj.time.dt.month.values]
    return obj.assign_coords(estacion=('time', etiquetas))

### Medias mensuales, estacionales y anuales

Cada producto es un `Dataset` con `tmedia`, `tmax`, `tmin` y `dtr` (medias del período).

| Producto | Frecuencia pandas | Etiqueta temporal |
|---|---|---|
| `t_mes` | `MS` | primer día del mes |
| `t_est` | `QS-DEC` (DJF, MAM, JJA, SON) | primer día del trimestre (DJF se etiqueta en diciembre) |
| `t_anio` | `YS` | 1-ene del año |

In [ ]:
t_mes = remuestrear(t2m, 'MS')
t_est = etiquetar_estacion(remuestrear(t2m, 'QS-DEC'))
t_anio = remuestrear(t2m, 'YS')

for nombre, obj in {'mensual': t_mes, 'estacional': t_est, 'anual': t_anio}.items():
    t = pd.DatetimeIndex(obj.time.values)
    print(f'{nombre:12s} {len(t):4d} períodos completos | {t[0].strftime("%Y-%m-%d")} -> {t[-1].strftime("%Y-%m-%d")}')

meses_ok = pd.DatetimeIndex(t_mes.time.values)
anios_ok = pd.DatetimeIndex(t_anio.time.values)

## 4. Índices de temperatura (tipo ETCCDI) y grados-día

Índices anuales calculados por celda, con los umbrales del bloque de parámetros:

| Variable | Índice | Definición |
|---|---|---|
| `txx`, `txn` | TXx, TXn | máximo y mínimo anual de la temperatura máxima diaria |
| `tnx`, `tnn` | TNx, TNn | máximo y mínimo anual de la temperatura mínima diaria |
| `fd` | FD | días con helada (TN < 0 °C) |
| `id` | ID | días de hielo (TX < 0 °C) |
| `su` | SU | días de verano (TX > 25 °C) |
| `tr` | TR | noches tropicales (TN > 20 °C) |
| `tx90p` | TX90p | % de días con TX sobre el percentil 90 del mes calendario (2000-2025) |
| `tn10p` | TN10p | % de días con TN bajo el percentil 10 del mes calendario (2000-2025) |
| `racha_calida` | ~WSDI | racha máxima anual de días consecutivos con TX > P90 |
| `racha_heladas` | ~CFD | racha máxima anual de días consecutivos con helada |
| `gdd` | GDD | grados-día de crecimiento: Σ max(tmedia − base, 0) |
| `hdd` | HDD | grados-día de calefacción: Σ max(base − tmedia, 0) |

> Simplificación: ETCCDI define los percentiles con una ventana móvil de 5 días centrada en cada día del año; aquí se usa el **mes calendario** como ventana
> (misma idea, umbral escalonado). Con TX90p/TN10p el valor medio es ~10 % por construcción: lo interesante es su evolución en el tiempo.

Docs: [`DataArray.groupby`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.groupby.html),
[`DataArray.quantile`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.quantile.html),
[`DataArray.clip`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.clip.html).

In [ ]:
tx, tn, tm = t2m.tmax, t2m.tmin, t2m.tmedia
valido = tm.notnull()


def conteo_anual(condicion):
    """Número de días por año en que se cumple `condicion` (NaN fuera de Chile)."""
    return remuestrear(condicion.where(valido), 'YS', como='sum')


def porcentaje_anual(condicion):
    """% de días del año en que se cumple `condicion`."""
    return 100 * remuestrear(condicion.where(valido), 'YS')


txx, txn = remuestrear(tx, 'YS', como='max'), remuestrear(tx, 'YS', como='min')
tnx, tnn = remuestrear(tn, 'YS', como='max'), remuestrear(tn, 'YS', como='min')

fd = conteo_anual(tn < UMBRAL_HELADA)
id_ = conteo_anual(tx < UMBRAL_HELADA)
su = conteo_anual(tx > UMBRAL_SU)
tr = conteo_anual(tn > UMBRAL_TR)

gdd = remuestrear((tm - BASE_GDD).clip(min=0), 'YS', como='sum')
hdd = remuestrear((BASE_HDD - tm).clip(min=0), 'YS', como='sum')

# Percentiles por mes calendario (referencia: todo el período). Requiere un único bloque en time.
p90_tx = tx.groupby('time.month').quantile(PERCENTIL_CALIDO, dim='time', skipna=True).drop_vars('quantile')
p10_tn = tn.groupby('time.month').quantile(PERCENTIL_FRIO, dim='time', skipna=True).drop_vars('quantile')
dia_calido = (tx.groupby('time.month') - p90_tx) > 0
noche_fria = (tn.groupby('time.month') - p10_tn) < 0

tx90p = porcentaje_anual(dia_calido)
tn10p = porcentaje_anual(noche_fria)

### Rachas máximas (días cálidos y heladas consecutivas)

Para cada año se recorre la serie diaria: `cumsum` de días que cumplen la condición menos el valor acumulado en el último día que no la cumple da la longitud de la racha vigente;
su máximo es la racha anual. La función trabaja sobre un eje de tiempo por año (no cruza el 31-dic → 1-ene) y se aplica bloque a bloque con `apply_ufunc(dask='parallelized')`.
Docs: [`xarray.apply_ufunc`](https://docs.xarray.dev/en/stable/generated/xarray.apply_ufunc.html).

In [ ]:
def _racha_max(cond):
    """cond: bool con el tiempo en el último eje. Devuelve la racha máxima de Trues."""
    acumulado = np.cumsum(cond, axis=-1)
    reinicio = np.maximum.accumulate(np.where(cond, 0, acumulado), axis=-1)
    return (acumulado - reinicio).max(axis=-1).astype('float32')


def racha_anual(cond):
    def _por_anio(x):
        return xr.apply_ufunc(_racha_max, x, input_core_dims=[['time']],
                              dask='parallelized', output_dtypes=[np.float32])
    # las operaciones groupby por mes (p. ej. dia_calido) dejan el tiempo en varios bloques: se vuelve a uno solo
    out = cond.chunk(time=-1).groupby('time.year').map(_por_anio)
    fechas = pd.to_datetime([f'{a}-01-01' for a in out.year.values])
    return out.assign_coords(year=fechas).rename(year='time')


hay_datos = valido.any('time')
racha_calida = racha_anual(dia_calido.where(valido, False)).where(hay_datos).sel(time=anios_ok)
racha_heladas = racha_anual((tn < UMBRAL_HELADA).where(valido, False)).where(hay_datos).sel(time=anios_ok)

indices = xr.Dataset({
    'txx': txx, 'txn': txn, 'tnx': tnx, 'tnn': tnn,
    'fd': fd, 'id': id_, 'su': su, 'tr': tr,
    'tx90p': tx90p, 'tn10p': tn10p, 'racha_calida': racha_calida, 'racha_heladas': racha_heladas,
    'gdd': gdd, 'hdd': hdd,
})

## 5. Climatologías y anomalías

- **Climatología** = media del período 2000-2025 de las medias (mensual por mes del año, estacional por estación, anual).
- **Anomalía absoluta** (°C) para las cuatro variables y **estandarizada** (z = anomalía / desviación estándar) para `tmedia`.
  Para temperatura no se usa anomalía en %: la escala en °C no tiene cero absoluto útil.
- **Amplitud del ciclo anual** = mes más cálido − mes más frío de la climatología de `tmedia` (alta en el interior, baja en la costa).

Docs: [`Dataset.groupby`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.groupby.html).

In [ ]:
def con_prefijo(ds_, prefijo):
    """Renombra las variables de un Dataset: tmedia -> {prefijo}tmedia."""
    return ds_.rename({v: f'{prefijo}{v}' for v in ds_.data_vars})


# --- Mensual ---
g_mes = t_mes.groupby('time.month')
clim_mes = g_mes.mean('time')
std_mes = g_mes.std('time')
anom_mes = g_mes - clim_mes
z_mes = (anom_mes.groupby('time.month') / std_mes.where(std_mes > 0)).tmedia

# --- Estacional ---
g_est = t_est.groupby('estacion')
clim_est = g_est.mean('time').sel(estacion=ORDEN_ESTACIONES)
anom_est = g_est - g_est.mean('time')

# --- Anual ---
clim_anio, std_anio = t_anio.mean('time'), t_anio.std('time')
anom_anio = t_anio - clim_anio
z_anio = (anom_anio / std_anio.where(std_anio > 0)).tmedia

amplitud_anual = clim_mes.tmedia.max('month') - clim_mes.tmedia.min('month')

## 6. Agregación espacial

### Zonas de Chile

Media **ponderada por área** (peso ∝ cos(latitud), porque las celdas de 0,25° se achican hacia los polos) dentro de cada banda de latitud,
más el total de Chile. La media areal de una media temporal es la media temporal de la media areal.

> Ojo: cada banda de latitud mezcla costa, valle central y cordillera; la media zonal de temperatura queda fuertemente influida por la
> fracción de celdas andinas (más frías). Sirve para ver variabilidad y tendencias, no como "temperatura típica" de la zona.

Docs: [`Dataset.weighted`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.weighted.html).

In [ ]:
lat = t2m.latitude
mascaras = [mascara_chile & (lat <= n) & (lat > s) for n, s in ZONAS.values()] + [mascara_chile]
mascaras_zonas = xr.concat(mascaras, dim=pd.Index(list(ZONAS) + [NOMBRE_TOTAL], name='zona'))
pesos_zonas = np.cos(np.deg2rad(lat)) * mascaras_zonas


def media_areal(obj):
    """Media ponderada por área en cada zona (dimensión nueva `zona`)."""
    return obj.weighted(pesos_zonas).mean(('latitude', 'longitude'))


print('Celdas por zona:', {z: int(mascaras_zonas.sel(zona=z).sum()) for z in mascaras_zonas.zona.values})

zonas_mes = media_areal(t_mes)
zonas_est = media_areal(t_est)
zonas_anio = media_areal(t_anio)
zonas_indices = media_areal(indices)

### Ciudades

Se toma la celda ERA5 más cercana a cada ciudad con selección puntual ([`DataArray.sel(method='nearest')`](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.sel.html)),
sobre la serie **sin máscara** (`t2m_raw`). Ojo: ERA5 tiene resolución de ~25 km y la altura de la celda puede diferir bastante de la de la ciudad
(≈ −6,5 °C por km de desnivel); en ciudades costeras la celda puede ser mixta mar-tierra y suavizar la amplitud diaria.

In [ ]:
lat_c = xr.DataArray([v[0] for v in CIUDADES.values()], dims='ciudad', coords={'ciudad': list(CIUDADES)})
lon_c = xr.DataArray([v[1] for v in CIUDADES.values()], dims='ciudad', coords={'ciudad': list(CIUDADES)})
t_ciu = t2m_raw.sel(latitude=lat_c, longitude=lon_c, method='nearest')

ciudades_mes = remuestrear(t_ciu, 'MS')
ciudades_anio = remuestrear(t_ciu, 'YS')

## 7. Cómputo y guardado

Se arman los `Dataset` de salida (todos siguen siendo perezosos) y recién aquí se ejecuta el cálculo: cada producto se computa y se escribe a NetCDF
por separado, y las series de zonas/ciudades se traen a memoria. Un grafo por producto (en vez de uno gigante que los una a todos) mantiene el uso de memoria bajo;
el costo es leer el diario del disco una vez por producto. Cada producto se calcula primero en memoria (`.compute()`) y después se escribe:
escribir directamente desde Dask con `to_netcdf` serializa la escritura y resulta mucho más lento.
Docs: [`Dataset.to_netcdf`](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.to_netcdf.html), [`dask.compute`](https://docs.dask.org/en/stable/api.html#dask.compute).

In [ ]:
ATTRS_COMUNES = {
    'source': 'ERA5 (ARCO, Google Cloud) - t2m diaria agregada en era5_agregacion_temperatura.ipynb',
    'periodo': f'{ANIO_INI}-{ANIO_FIN}',
    'mascara_chile': str(bool(USAR_MASCARA_CHILE) and int(mascara_chile.sum()) < mascara_chile.size),
    'frontera_dia': 'UTC+0',
    'nota': 'Solo períodos completos; medias de los valores diarios (°C).',
}
UNIDADES_INDICES = {
    'fd': 'días', 'id': 'días', 'su': 'días', 'tr': 'días', 'tx90p': '%', 'tn10p': '%',
    'racha_calida': 'días', 'racha_heladas': 'días', 'gdd': '°C·día', 'hdd': '°C·día',
}


def con_attrs(ds_, titulo, unidades=None):
    """Título, atributos comunes y unidades (°C por defecto)."""
    ds_.attrs = {'title': titulo, **ATTRS_COMUNES}
    unidades = unidades or {}
    for v in ds_.data_vars:
        ds_[v].attrs['units'] = unidades.get(v, '°C')
    return ds_


ds_mensual = con_attrs(xr.merge([
    t_mes, con_prefijo(anom_mes, 'anom_'), z_mes.rename('z_tmedia'),
    remuestrear((tn < UMBRAL_HELADA).where(valido), 'MS', como='sum').rename('fd'),
]), 'Temperatura mensual (media, máx, mín, amplitud diaria), anomalías y heladas',
    {'z_tmedia': '1', 'fd': 'días'})

ds_estacional = con_attrs(xr.merge([t_est, con_prefijo(anom_est, 'anom_')]),
                          'Temperatura estacional (DJF, MAM, JJA, SON)')

ds_anual = con_attrs(xr.merge([
    t_anio, con_prefijo(anom_anio, 'anom_'), z_anio.rename('z_tmedia'), indices,
]), 'Temperatura anual, anomalías e índices de extremos', {'z_tmedia': '1', **UNIDADES_INDICES})

ds_clim = con_attrs(xr.merge([
    con_prefijo(clim_mes, 'clim_mes_'), con_prefijo(std_mes, 'std_mes_'), con_prefijo(clim_est, 'clim_est_'),
    con_prefijo(clim_anio, 'clim_anio_'), con_prefijo(std_anio, 'std_anio_'),
    amplitud_anual.rename('amplitud_anual'), p90_tx.rename('p90_tx'), p10_tn.rename('p10_tn'),
]), f'Climatologías {ANIO_INI}-{ANIO_FIN}')

PRODUCTOS = {
    't2m_mensual': ds_mensual, 't2m_estacional': ds_estacional, 't2m_anual': ds_anual, 't2m_climatologias': ds_clim,
}
SERIES = {
    'zonas_mensual': zonas_mes, 'zonas_estacional': zonas_est, 'zonas_anual': zonas_anio, 'zonas_indices': zonas_indices,
    'ciudades_mensual': ciudades_mes, 'ciudades_anual': ciudades_anio,
}


def escribir(ds_, nombre):
    ds_ = ds_.compute()   # cálculo en el cluster; el producto cabe en memoria
    enc = {v: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for v in ds_.data_vars}
    ds_.to_netcdf(DIR_SALIDA / f'{nombre}.nc', engine='netcdf4', encoding=enc)
    print(f'{nombre}.nc  {(DIR_SALIDA / (nombre + ".nc")).stat().st_size / 1e6:.1f} MB')

In [ ]:
for nombre, producto in PRODUCTOS.items():
    escribir(producto, nombre)

# Series de zonas y ciudades: pequeñas, se traen a memoria
series = dict(zip(SERIES, dask.compute(*SERIES.values())))
print('Series en memoria:', list(series))

### Relectura de los productos

Ahora los productos son pequeños comparados con el diario: se cargan en memoria y se cierra el archivo
([`xarray.load_dataset`](https://docs.xarray.dev/en/stable/generated/xarray.load_dataset.html)) para no bloquear la escritura si se re-ejecuta la sección 7 en Windows.

In [ ]:
mensual, estacional, anual, clim = (xr.load_dataset(DIR_SALIDA / f'{n}.nc') for n in PRODUCTOS)
z_mes_s, z_est_s, z_anio_s, z_ind_s = (series[k] for k in ['zonas_mensual', 'zonas_estacional', 'zonas_anual', 'zonas_indices'])
c_mes_s, c_anio_s = series['ciudades_mensual'], series['ciudades_anual']

print('Mensual  :', dict(mensual.sizes), '| variables:', list(mensual.data_vars))
print('Anual    :', dict(anual.sizes), '| variables:', list(anual.data_vars))
print('Zonas    :', list(z_anio_s.zona.values))

## 8. Validación

Comprobaciones de coherencia:
1. La media de los 12 meses de cada año, **ponderada por el número de días de cada mes**, coincide con la media anual (en cada celda).
2. La media de las cuatro estaciones consecutivas (DJF que empieza en dic Y-1, MAM, JJA, SON de Y), ponderada por días, coincide con la de esos mismos 12 meses (dic Y-1 a nov Y).
3. `tmin ≤ tmedia ≤ tmax` en las medias mensuales y anuales, y `TNn ≤ tmin anual`, `tmax anual ≤ TXx`.
4. La media areal de Chile queda entre los mínimos y máximos de las zonas.
5. Coherencia de índices: ID ≤ FD, conteos entre 0 y 366, TX90p y TN10p entre 0 y 100 %.

In [ ]:
VARS_BASE = ['tmedia', 'tmax', 'tmin']
TOL = 1e-3   # °C (los productos se guardan en float32)

# 1) meses vs año (celda a celda), media ponderada por días
dias_mes = mensual.time.dt.days_in_month
media_meses = (mensual[VARS_BASE] * dias_mes).resample(time='YS').sum(min_count=12) / dias_mes.resample(time='YS').sum()
comunes = media_meses.time.to_index().intersection(anual.time.to_index())
assert len(comunes) > 0
dif = max(float(np.abs(media_meses[v].sel(time=comunes) - anual[v].sel(time=comunes)).max()) for v in VARS_BASE)
print(f'1) máx |media ponderada de meses - media anual| = {dif:.2e} °C  ({len(comunes)} años)')
assert dif < TOL

# 2) estaciones vs meses: DJF de dic(Y-1) + MAM + JJA + SON de Y  ==  dic(Y-1) ... nov(Y)
est = estacional[VARS_BASE]
t_est_idx = est.time.to_index()
dias_est = xr.DataArray(((t_est_idx + pd.DateOffset(months=3)) - t_est_idx).days, dims='time', coords={'time': est.time})
t_mes_idx = mensual.time.to_index()
ok = 0
for anio in range(ANIO_INI + 1, ANIO_FIN + 1):
    t_ini = [pd.Timestamp(anio - 1, 12, 1), pd.Timestamp(anio, 3, 1), pd.Timestamp(anio, 6, 1), pd.Timestamp(anio, 9, 1)]
    t_12 = pd.date_range(pd.Timestamp(anio - 1, 12, 1), periods=12, freq='MS')
    if all(t in t_est_idx for t in t_ini) and all(t in t_mes_idx for t in t_12):
        w_e, w_m = dias_est.sel(time=t_ini), dias_mes.sel(time=t_12)
        m_est = (est.sel(time=t_ini) * w_e).sum('time') / w_e.sum()
        m_mes = (mensual[VARS_BASE].sel(time=t_12) * w_m).sum('time') / w_m.sum()
        dif_e = max(float(np.abs(m_est[v] - m_mes[v]).max()) for v in VARS_BASE)
        assert dif_e < TOL, f'Estaciones != meses en {anio}'
        ok += 1
print(f'2) estaciones vs meses verificadas en {ok} años')
assert ok > 0


def menor_igual(a, b, tol=TOL):
    """a <= b en todas las celdas (las enmascaradas son NaN y se ignoran)."""
    return bool(((a <= b + tol) | a.isnull() | b.isnull()).all())


# 3) orden físico de las variables
for prod in (mensual, anual):
    assert menor_igual(prod.tmin, prod.tmedia) and menor_igual(prod.tmedia, prod.tmax)
assert menor_igual(anual.tnn, anual.tmin) and menor_igual(anual.tmax, anual.txx)
assert menor_igual(anual.tnn, anual.tnx) and menor_igual(anual.txn, anual.txx)
print('3) tmin <= tmedia <= tmax y extremos coherentes con las medias')

# 4) coherencia espacial
chile = z_anio_s.tmedia.sel(zona=NOMBRE_TOTAL)
zonas_solo = z_anio_s.tmedia.sel(zona=list(ZONAS))
assert bool((chile <= zonas_solo.max('zona') + 1e-6).all()) and bool((chile >= zonas_solo.min('zona') - 1e-6).all())
print('4) la media de Chile queda dentro del rango de las zonas')

# 5) coherencia de índices
assert menor_igual(anual['id'], anual.fd)
for v in ['fd', 'id', 'su', 'tr', 'racha_calida', 'racha_heladas']:
    assert float(anual[v].min()) >= 0 and float(anual[v].max()) <= 366, v
for v in ['tx90p', 'tn10p']:
    assert float(anual[v].min()) >= 0 and float(anual[v].max()) <= 100, v
assert menor_igual(anual.racha_heladas, anual.fd)
print('5) índices coherentes. Validación OK.')

## 9. Visualizaciones

Helper de mapas: `pcolormesh` de la grilla sin proyección (cada celda ≈ 25 km) con el contorno de Chile / límites regionales encima (`contorno`, sección 2). Docs: [`matplotlib.axes.Axes.pcolormesh`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.pcolormesh.html).

In [ ]:
def mapa(da, ax, cmap, titulo, etiqueta, vmin=None, vmax=None):
    im = ax.pcolormesh(da.longitude, da.latitude, da.values, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto')
    if contorno is not None:
        contorno.plot(ax=ax, color='#333333', linewidth=0.5)
        ax.set_ylabel('')   # geopandas pone su propia etiqueta de eje
    ax.set_xlim(float(da.longitude.min()) - 0.125, float(da.longitude.max()) + 0.125)
    ax.set_ylim(float(da.latitude.min()) - 0.125, float(da.latitude.max()) + 0.125)
    ax.set_aspect('equal')
    ax.set_title(titulo)
    ax.set_xlabel('Longitud (°)')
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.55, pad=0.03, label=etiqueta)
    return im


def p_alto(da, q=98):
    return float(np.nanpercentile(np.asarray(da), q))


def p_bajo(da, q=2):
    return float(np.nanpercentile(np.asarray(da), q))

### 9.1 Mapas de la climatología anual

Temperatura media, máxima y mínima medias anuales (misma escala de color para las tres) y amplitud del ciclo anual (mes más cálido − mes más frío).

In [ ]:
vmin_t, vmax_t = p_bajo(clim.clim_anio_tmin), p_alto(clim.clim_anio_tmax)
fig, axs = plt.subplots(1, 4, figsize=(14, 10), constrained_layout=True, sharey=True)
mapa(clim.clim_anio_tmedia, axs[0], CMAP_T, 'Temperatura media anual', '°C', vmin_t, vmax_t)
mapa(clim.clim_anio_tmax, axs[1], CMAP_T, 'Máxima media anual', '°C', vmin_t, vmax_t)
mapa(clim.clim_anio_tmin, axs[2], CMAP_T, 'Mínima media anual', '°C', vmin_t, vmax_t)
mapa(clim.amplitud_anual, axs[3], 'YlOrRd', 'Amplitud del ciclo anual', '°C', 0, p_alto(clim.amplitud_anual))
axs[0].set_ylabel('Latitud (°)')
fig.suptitle(f'ERA5 · Chile {ANIO_INI}-{ANIO_FIN}', fontsize=13)
guardar_figura(fig, '01_mapas_climatologia_anual')
plt.show()

### 9.2 Climatología estacional

Temperatura media por estación con una misma escala de color para comparar (DJF = dic-ene-feb, etc.).

In [ ]:
vmin_est, vmax_est = p_bajo(clim.clim_est_tmedia, 1), p_alto(clim.clim_est_tmedia, 99)
fig, axs = plt.subplots(1, 4, figsize=(14, 9), constrained_layout=True, sharey=True)
for ax, est_ in zip(axs, ORDEN_ESTACIONES):
    mapa(clim.clim_est_tmedia.sel(estacion=est_), ax, CMAP_T, est_, '°C', vmin_est, vmax_est)
axs[0].set_ylabel('Latitud (°)')
guardar_figura(fig, '02_mapas_climatologia_estacional')
plt.show()

### 9.3 Ciclo anual por zona

Media mensual 2000-2025 de `tmax`, `tmedia` y `tmin` (banda sombreada entre mínima y máxima) con ±1 desviación estándar interanual de `tmedia`.
Cada zona con su propio eje Y.

In [ ]:
zonas_plot = list(ZONAS) + [NOMBRE_TOTAL]
meses = np.arange(1, 13)
fig, axs = plt.subplots(2, 3, figsize=(12, 6.5), constrained_layout=True, sharex=True)
for ax, z in zip(axs.flat, zonas_plot):
    g = z_mes_s.sel(zona=z).groupby('time.month')
    m, s = g.mean('time'), g.std('time')
    ax.fill_between(meses, m.tmin, m.tmax, color=COLOR_TMAX, alpha=0.12, lw=0)
    ax.plot(meses, m.tmax, color=COLOR_TMAX, lw=1.4, label='máx')
    ax.errorbar(meses, m.tmedia, yerr=s.tmedia, color=COLOR_TMEDIA, lw=1.6, elinewidth=0.8, capsize=2, label='media')
    ax.plot(meses, m.tmin, color=COLOR_TMIN, lw=1.4, label='mín')
    ax.set_title(z)
    ax.set_ylabel('°C')
    ax.set_xticks(meses, NOMBRES_MES, rotation=90)
axs.flat[0].legend(frameon=False, fontsize=8, loc='lower left')
guardar_figura(fig, '03_ciclo_anual_zonas')
plt.show()

### 9.4 Anomalía anual por zona

Anomalía de la temperatura media anual respecto de 2000-2025, en °C (mismo eje para todas las zonas, por eso son comparables).
Rojo = año más cálido que lo normal; azul = más frío.

In [ ]:
anom_zonas = z_anio_s.tmedia - z_anio_s.tmedia.mean('time')
anios = z_anio_s.time.dt.year.values
fig, axs = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True, sharex=True, sharey=True)
for ax, z in zip(axs.flat, zonas_plot):
    v = anom_zonas.sel(zona=z).values
    ax.bar(anios, v, color=np.where(v >= 0, COLOR_POS, COLOR_NEG), width=0.75)
    ax.axhline(0, color=COLOR_TEXTO, lw=0.8)
    ax.set_title(z)
for ax in axs[:, 0]:
    ax.set_ylabel('Anomalía (°C)')
guardar_figura(fig, '04_anomalia_anual_zonas')
plt.show()

### 9.5 Anomalías mensuales año × mes

Anomalía de la temperatura media respecto de la climatología de cada mes en la zona `ZONA_HEATMAP` (parámetro). Permite ver de un vistazo años/meses cálidos o fríos
(p. ej. inviernos con olas de frío o veranos con olas de calor).

In [ ]:
serie = z_mes_s.tmedia.sel(zona=ZONA_HEATMAP)
anom_z = serie.groupby('time.month') - serie.groupby('time.month').mean('time')

tabla = pd.Series(anom_z.values, index=pd.MultiIndex.from_arrays([serie.time.dt.year.values, serie.time.dt.month.values])).unstack()
tabla = tabla.reindex(columns=range(1, 13))
lim = float(np.nanpercentile(np.abs(tabla.values), 98))

fig, ax = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
im = ax.imshow(tabla.T.values, aspect='auto', cmap='RdBu_r', vmin=-lim, vmax=lim, origin='upper',
               extent=[tabla.index.min() - 0.5, tabla.index.max() + 0.5, 12.5, 0.5])
ax.set_yticks(range(1, 13), NOMBRES_MES)
ax.grid(False)
ax.set_title(f'Anomalía mensual de temperatura media (°C) · {ZONA_HEATMAP}')
plt.colorbar(im, ax=ax, pad=0.02, label='°C')
guardar_figura(fig, '05_heatmap_anomalias_mensuales')
plt.show()

### 9.6 Ciclo anual de ciudades

Ciclo anual medio de la celda ERA5 más cercana: banda entre mínima y máxima medias, línea de la media. Mismo eje Y para comparar entre ciudades.
En el título, la temperatura media anual; en el recuadro, las coordenadas reales de la celda usada.

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(12, 9), constrained_layout=True, sharex=True, sharey=True)
for ax, ciudad in zip(axs.flat, CIUDADES):
    s = c_mes_s.sel(ciudad=ciudad)
    m = s.groupby('time.month').mean('time')
    ax.fill_between(meses, m.tmin, m.tmax, color=COLOR_TMAX, alpha=0.15, lw=0)
    ax.plot(meses, m.tmax, color=COLOR_TMAX, lw=1.2)
    ax.plot(meses, m.tmedia, color=COLOR_TMEDIA, lw=1.6)
    ax.plot(meses, m.tmin, color=COLOR_TMIN, lw=1.2)
    media = float(c_anio_s.tmedia.sel(ciudad=ciudad).mean('time'))
    ax.set_title(f'{ciudad} · {media:.1f} °C'.replace('.', ','))
    ax.set_xticks(meses, NOMBRES_MES, rotation=90)
    ax.text(0.03, 0.95, f'celda {float(s.latitude):.2f}°, {float(s.longitude):.2f}°', transform=ax.transAxes,
            ha='left', va='top', fontsize=8, color=COLOR_TEXTO)
for ax in axs[:, 0]:
    ax.set_ylabel('°C')
guardar_figura(fig, '06_ciclo_anual_ciudades')
plt.show()

### 9.7 Índices de extremos (media 2000-2025)

Promedio anual de los índices de la sección 4 para cada celda. TR (noches tropicales) no se mapea: en ERA5 es prácticamente cero en todo Chile.

In [ ]:
mapas_indices = [
    ('fd', f'FD · heladas (TN < {UMBRAL_HELADA:g} °C)', 'días/año', 'Blues', 0, None),
    ('su', f'SU · días > {UMBRAL_SU:g} °C', 'días/año', 'OrRd', 0, None),
    ('hdd', f'HDD · grados-día calefacción base {BASE_HDD:g} °C', '°C·día', 'PuBu', 0, None),
    ('txx', 'TXx · máxima absoluta', '°C', CMAP_T, None, None),
    ('tnn', 'TNn · mínima absoluta', '°C', CMAP_T, None, None),
    ('gdd', f'GDD · grados-día base {BASE_GDD:g} °C', '°C·día', 'YlGn', 0, None),
]
fig, axs = plt.subplots(1, 6, figsize=(20, 9), constrained_layout=True, sharey=True)
for ax, (v, titulo, unidad, cmap, vmin, vmax) in zip(axs, mapas_indices):
    d = anual[v].mean('time')
    mapa(d, ax, cmap, titulo, unidad,
         p_bajo(d) if vmin is None else vmin, p_alto(d) if vmax is None else vmax)
    ax.title.set_fontsize(9)
axs[0].set_ylabel('Latitud (°)')
guardar_figura(fig, '07_mapas_indices_extremos')
plt.show()

### 9.8 Serie diaria en una celda

Temperatura diaria 2000-2025 de la celda ERA5 más cercana a la ciudad elegida en `CIUDAD_SERIE`: banda diaria mín-máx y media móvil de 30 días de `tmedia`.

In [ ]:
CIUDAD_SERIE = 'Santiago'

serie_dia = t2m_raw.sel(latitude=CIUDADES[CIUDAD_SERIE][0], longitude=CIUDADES[CIUDAD_SERIE][1], method='nearest').compute()

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
ax.fill_between(serie_dia.time, serie_dia.tmin, serie_dia.tmax, color=COLOR_TMAX, alpha=0.2, lw=0, label='mín-máx diaria')
ax.plot(serie_dia.time, serie_dia.tmedia.rolling(time=30, center=True).mean(), color=COLOR_TMEDIA, lw=1.0,
        label='media (móvil 30 días)')
ax.set_title(f'Temperatura diaria {ANIO_INI}-{ANIO_FIN} · {CIUDAD_SERIE} '
             f'(celda {float(serie_dia.latitude):.2f}°, {float(serie_dia.longitude):.2f}°)')
ax.set_xlabel('Tiempo')
ax.set_ylabel('Temperatura [°C]')
ax.legend(frameon=False, loc='lower left')
guardar_figura(fig, '09_serie_diaria_ciudad')
plt.show()

## 10. Tendencias

Tendencia lineal (mínimos cuadrados) de las medias anuales de `tmedia`, `tmax` y `tmin`, expresada en **°C por década**, con valor p de dos colas de la pendiente (t de Student, n−2 grados de libertad).
Se usa `xarray.cov` y `xarray.corr`, sin bucles por celda. Docs: [`xarray.cov`](https://docs.xarray.dev/en/stable/generated/xarray.cov.html),
[`scipy.special.stdtr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.stdtr.html).

> Con 26 años, la variabilidad interanual (ENSO, PDO) pesa mucho. En Chile se espera enfriamiento o estancamiento de las máximas en la costa central-norte
> (surgencia costera) y calentamiento en el interior y la cordillera. No asume autocorrelación de los residuos: es una primera señal, no una atribución.

In [ ]:
def tendencia_lineal(da, dim='time'):
    """Pendiente por década, valor p y n de una regresión lineal contra el año."""
    t = xr.DataArray(da[dim].dt.year.values.astype(float), dims=dim, coords={dim: da[dim]})
    n = da.sizes[dim]
    pendiente = xr.cov(t, da, dim=dim) / t.var(dim, ddof=1)
    r = xr.corr(t, da, dim=dim)
    with np.errstate(divide='ignore', invalid='ignore'):
        t_stat = r * np.sqrt((n - 2) / (1 - r ** 2))
        p = 2 * xr.apply_ufunc(stdtr, n - 2, -np.abs(t_stat))
    return (pendiente * 10).rename('pendiente_decada'), p.rename('p_valor')


tendencias = {v: tendencia_lineal(anual[v]) for v in VARS_BASE}
lim = max(p_alto(np.abs(pend)) for pend, _ in tendencias.values())
titulos = {'tmedia': 'T media', 'tmax': 'T máxima', 'tmin': 'T mínima'}

fig, axs = plt.subplots(1, 3, figsize=(11, 10), constrained_layout=True, sharey=True)
for ax, (v, (pend, pval)) in zip(axs, tendencias.items()):
    mapa(pend, ax, 'RdBu_r', f'Tendencia {titulos[v]}', '°C/década', -lim, lim)
    sig = (pval < 0.05) & pend.notnull()
    yy, xx = np.meshgrid(pend.latitude.values, pend.longitude.values, indexing='ij')
    ax.scatter(xx[sig.values], yy[sig.values], s=3, color='k', label='p < 0,05')
    n_val = int(pend.notnull().sum())
    print(f'{v:7s} celdas con tendencia significativa (p<0,05): {int(sig.sum())} de {n_val} '
          f'({100 * int(sig.sum()) / n_val:.1f} %) | pendiente mediana: {float(pend.median()):+.3f} °C/década')
axs[0].legend(loc='lower left', frameon=False)
axs[0].set_ylabel('Latitud (°)')
fig.suptitle(f'Tendencia lineal {ANIO_INI}-{ANIO_FIN}', fontsize=13)
guardar_figura(fig, '08_tendencia_anual')
plt.show()

## 11. Tabla resumen y exportación a CSV

Resumen por zona con: temperaturas medias anuales, amplitud térmica diaria y del ciclo anual, variabilidad interanual, tendencias (°C/década con su valor p),
días de helada y de verano medios, y años extremos.
Las series agregadas se exportan a CSV en formato largo (UTF-8 con BOM, abre bien en Excel).

In [ ]:
tend_z = {v: tendencia_lineal(z_anio_s[v]) for v in VARS_BASE}
clim_mes_z = z_mes_s.tmedia.groupby('time.month').mean('time')
anio_num = z_anio_s.time.dt.year.values

resumen = pd.DataFrame({
    'tmedia_C': z_anio_s.tmedia.mean('time').to_series(),
    'tmax_C': z_anio_s.tmax.mean('time').to_series(),
    'tmin_C': z_anio_s.tmin.mean('time').to_series(),
    'dtr_C': z_anio_s.dtr.mean('time').to_series(),
    'amplitud_anual_C': (clim_mes_z.max('month') - clim_mes_z.min('month')).to_series(),
    'desv_est_anual_C': z_anio_s.tmedia.std('time').to_series(),
    'tend_tmedia_C_decada': tend_z['tmedia'][0].to_series(),
    'p_tmedia': tend_z['tmedia'][1].to_series(),
    'tend_tmax_C_decada': tend_z['tmax'][0].to_series(),
    'p_tmax': tend_z['tmax'][1].to_series(),
    'tend_tmin_C_decada': tend_z['tmin'][0].to_series(),
    'p_tmin': tend_z['tmin'][1].to_series(),
    'heladas_dias': z_ind_s.fd.mean('time').to_series(),
    'dias_verano': z_ind_s.su.mean('time').to_series(),
    'anio_mas_calido': pd.Series(anio_num[z_anio_s.tmedia.argmax('time').values], index=z_anio_s.zona.values),
    'anio_mas_frio': pd.Series(anio_num[z_anio_s.tmedia.argmin('time').values], index=z_anio_s.zona.values),
}).loc[zonas_plot].round(3)
resumen.index.name = 'zona'
resumen

In [ ]:
for nombre, serie_ in series.items():
    serie_.to_dataframe().to_csv(DIR_SALIDA / f'{nombre}.csv', encoding='utf-8-sig', float_format='%.3f')
resumen.to_csv(DIR_SALIDA / 'resumen_zonas.csv', encoding='utf-8-sig')

print('Archivos en', DIR_SALIDA)
for f in sorted(DIR_SALIDA.glob('*.*')):
    print(f'  {f.name:32s} {f.stat().st_size / 1e6:8.2f} MB')

### Cerrar el cluster

In [ ]:
client.close()
print('Cluster cerrado.')